<div style="background: linear-gradient(135deg, #0B1F3F 0%, #008C8C 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: white; font-family: Georgia, serif; margin: 0; font-size: 2.4em;">
🔍 Session 13B: Data Deep Dive
</h1>
<p style="color: #B8953E; font-size: 1.3em; margin-top: 10px; font-family: Calibri, sans-serif;">
Bivariate Analysis & Feature–Target Relationships
</p>
<p style="color: #B0D0D0; font-size: 0.95em; margin-top: 8px;">
Credit Risk Modelling Programme &nbsp;|&nbsp; MBA Advanced Analytics &nbsp;|&nbsp; 2025–26
</p>
</div>

<div style="background: #FFFFFF; border: 2px solid #B8953E; padding: 20px 28px; border-radius: 10px; margin: 10px 0 20px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">📖 Where We Are in the Journey</h3>
<p style="color: #333; font-size: 1.05em; line-height: 1.7;">
In <strong>Session 13</strong>, we mapped the data universe: shapes, types, missingness, schemas, and basic KPIs.
We know <em>what</em> the data looks like. Now we need to understand <em>how features relate to our target</em> —
who defaults and who doesn’t, and which characteristics separate them.
</p>
<p style="color: #333; font-size: 1.05em; line-height: 1.7;">
This session is deliberately <strong>visual and intuitive</strong>. We will look at every important feature through
the lens of default risk before Session 14 introduces formal statistical methods. Think of this as building
your “gut feeling” for the data — the kind of instinct that lets a credit analyst glance at an application
and say <em>“this one worries me.”</em>
</p>
</div>

<div style="background: #FFFFFF; border: 2px solid #008C8C; padding: 20px 28px; border-radius: 10px; margin: 10px 0 20px 0;">
<h3 style="color: #0B1F3F; margin-top: 0;">🗺️ Notebook Roadmap</h3>
<ol style="color: #333; font-size: 1.05em; line-height: 1.8;">
<li><strong>Setup & Data Reload</strong> — Reload the cleaned data from Session 13</li>
<li><strong>Categorical Features vs Default</strong> — Default rates across every category</li>
<li><strong>Numeric Features vs Default</strong> — Box plots, KDE overlays, and statistical comparisons</li>
<li><strong>Outlier Detection & Treatment</strong> — Identifying and handling extreme values</li>
<li><strong>Feature Transformations</strong> — Log transforms, binning, and why they matter</li>
<li><strong>Cross-Feature Interactions</strong> — When two features together tell a richer story</li>
<li><strong>Correlation Landscape</strong> — A visual, intuitive introduction to feature relationships</li>
<li><strong>Feature Scorecard</strong> — Ranking features by their apparent predictive power</li>
</ol>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 1: Setup & Data Reload</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Pick up where Session 13 left off</p>
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# ── Branding colours ──
NAVY  = '#0B1F3F'
TEAL  = '#008C8C'
GOLD  = '#B8953E'
CORAL = '#E8634A'
LGOLD = '#FDF6E8'
LTEAL = '#E0F2F2'
PURPLE = '#6C5B7B'
PALETTE = [NAVY, TEAL, GOLD, CORAL, PURPLE, '#C06C84']
sns.set_palette(PALETTE)

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.titleweight': 'bold',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("\u2705 Libraries loaded.")

In [ ]:
# ── Load and clean data (reproducing Session 13 steps) ──
DATA_DIR = '../data/home-credit-default-risk/'

app = pd.read_csv(os.path.join(DATA_DIR, 'application_train.csv'))

# Clean DAYS_EMPLOYED anomaly (from Session 13)
app['DAYS_EMPLOYED_ANOMALY'] = (app['DAYS_EMPLOYED'] == 365243).astype(int)
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)

# Derive KPIs (from Session 13)
app['AGE_YEARS'] = (-app['DAYS_BIRTH'] / 365).round(1)
app['EMPLOYMENT_YEARS'] = (-app['DAYS_EMPLOYED'] / 365).round(1)
app['DEBT_TO_INCOME'] = (app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']).round(2)
app['ANNUITY_BURDEN'] = (app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']).round(4)
app['CREDIT_GOODS_RATIO'] = (app['AMT_CREDIT'] / app['AMT_GOODS_PRICE']).round(3)
app['EXT_SCORE_BLEND'] = app[
    ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
].mean(axis=1).round(4)

print(f"\u2705 Data loaded: {app.shape[0]:,} rows \u00d7 {app.shape[1]} columns")
print(f"   Default rate: {app['TARGET'].mean():.2%}")

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 2: Categorical Features vs Default</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Which categories carry more risk?</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">For every important categorical feature, compute the default rate within each category. Identify which categories are associated with higher or lower default risk. Learn to read a “default rate bar chart” — the single most common visual in credit analytics.</span>
</div>

<div style="background: #E0F2F2; border: 2px solid #008C8C; padding: 16px 22px; border-radius: 8px; margin: 12px 0;">
<strong style="color: #008C8C;">📐 What is a “Default Rate Bar Chart”?</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">
Instead of comparing raw counts (which are misleading because of class imbalance), we compute:
<br><br>
<code style="background: white; padding: 4px 8px; border-radius: 4px;">Default Rate = Number of defaults in category / Total applicants in category</code>
<br><br>
A category with a 15% default rate is roughly twice as risky as the overall 8% baseline.
The overall default rate is our <strong>benchmark line</strong> — categories above it are riskier, below it are safer.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 1: Default Rate by Contract Type</h3>
</div>

In [ ]:
# Default Rate by Contract Type
overall_rate = app['TARGET'].mean()

grouped = app.groupby('NAME_CONTRACT_TYPE')['TARGET'].agg(['mean', 'count']).reset_index()
grouped.columns = ['Contract Type', 'Default Rate', 'Count']
grouped = grouped.sort_values('Default Rate', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(grouped['Contract Type'], grouped['Default Rate'],
               color=[CORAL if r > overall_rate else TEAL for r in grouped['Default Rate']],
               edgecolor='white', height=0.5)
ax.axvline(overall_rate, color=GOLD, linestyle='--', linewidth=1.8, label=f'Overall rate: {overall_rate:.2%}')
for bar, rate, count in zip(bars, grouped['Default Rate'], grouped['Count']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{rate:.2%}  (n={count:,})', va='center', fontsize=10)
ax.set_xlabel('Default Rate')
ax.set_title('Default Rate by Contract Type', color=NAVY, fontweight='bold')
ax.legend()
ax.set_xlim(0, grouped['Default Rate'].max() * 1.35)
plt.tight_layout()
plt.show()
print(grouped.to_string(index=False))


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">The default rate IS the mean of TARGET within each group, since TARGET is 0/1. <code>app.groupby('NAME_CONTRACT_TYPE')['TARGET'].mean()</code> gives you default rates directly.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Revolving loans have a significantly higher default rate</strong> than cash loans. This is intuitive: revolving credit (like credit lines) tends to be used by borrowers who need ongoing access to funds, which can signal financial stress. A credit manager might apply stricter criteria for revolving loan applicants.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 2: Default Rates Across Key Categorical Features</h3>
</div>

In [ ]:
# Default Rates Across 6 Categorical Features
cat_features = [
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
]
overall_rate = app['TARGET'].mean()

def plot_default_rate(feature, ax, top_n=15):
    grouped = (app.groupby(feature)['TARGET']
               .agg(['mean', 'count'])
               .reset_index()
               .sort_values('mean', ascending=True))
    if len(grouped) > top_n:
        grouped = grouped.tail(top_n)
    colors = [CORAL if r > overall_rate else TEAL for r in grouped['mean']]
    ax.barh(grouped[feature], grouped['mean'], color=colors, edgecolor='white', height=0.6)
    ax.axvline(overall_rate, color=GOLD, linestyle='--', linewidth=1.5)
    ax.set_title(feature, color=NAVY, fontweight='bold', fontsize=11)
    ax.set_xlabel('Default Rate')
    ax.tick_params(axis='y', labelsize=8)

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.ravel()
for i, feat in enumerate(cat_features):
    plot_default_rate(feat, axes[i])
plt.suptitle('Default Rates by Categorical Features', fontsize=15, color=NAVY, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Create a reusable function! Define <code>plot_default_rate(feature, ax)</code> that takes a feature name and axis, computes <code>groupby(...).mean()</code>, and plots a horizontal bar chart. Then loop through features.</span>
</div>

<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;"><strong>Key patterns emerging:</strong><br>• <strong>Income type:</strong> “Maternity leave” and “Unemployed” show the highest default rates. “Pensioner” is the safest.<br>• <strong>Education:</strong> “Lower secondary” defaults most; “Academic degree” defaults least. Education is protective.<br>• <strong>Family status:</strong> “Civil marriage” and “Single” are riskier than “Married” or “Widow”.<br>• <strong>Occupation:</strong> “Low-skill labourers” and “Drivers” are the riskiest occupations.<br>These patterns make intuitive business sense. A stable job, higher education, and marriage are all markers of financial stability.</span>
</div>

<div style="background: #F0E6F6; border-left: 5px solid #6C5B7B; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #6C5B7B;">👔 Manager’s Take</strong><br>
<span style="color: #333;">As a lending manager, these charts are already actionable. You might, for example, flag applications from “Maternity leave” income types for additional review — not to deny them, but to assess whether the loan terms are appropriate. The goal is never to discriminate, but to ensure the loan sets the borrower up for success.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 3: Gender and Age Profile</h3>
</div>

In [ ]:
# Gender and Age Profile

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
overall_rate = app['TARGET'].mean()

# 1. Default rate by gender
gender_df = app[app['CODE_GENDER'] != 'XNA']
gender_grouped = gender_df.groupby('CODE_GENDER')['TARGET'].agg(['mean','count']).reset_index()
axes[0].bar(gender_grouped['CODE_GENDER'], gender_grouped['mean'],
            color=[TEAL, CORAL], edgecolor='white', width=0.4)
axes[0].axhline(overall_rate, color=GOLD, linestyle='--', linewidth=1.5, label=f'Overall: {overall_rate:.2%}')
for i, (rate, count) in enumerate(zip(gender_grouped['mean'], gender_grouped['count'])):
    axes[0].text(i, rate + 0.002, f'{rate:.2%}', ha='center', fontweight='bold', color=NAVY)
axes[0].set_title('Default Rate by Gender', color=NAVY, fontweight='bold')
axes[0].set_ylabel('Default Rate')
axes[0].legend()

# 2. Age density split by TARGET
for target_val, color, label in [(0, TEAL, 'Repaid'), (1, CORAL, 'Default')]:
    app[app['TARGET'] == target_val]['AGE_YEARS'].plot.kde(ax=axes[1], color=color, label=label, linewidth=2)
axes[1].set_title('Age Distribution by TARGET', color=NAVY, fontweight='bold')
axes[1].set_xlabel('Age (Years)')
axes[1].legend()

# 3. Default rate by age bin
age_bins = [20, 25, 30, 35, 40, 45, 50, 55, 60, 70]
app['AGE_BIN'] = pd.cut(app['AGE_YEARS'], bins=age_bins)
age_default = app.groupby('AGE_BIN')['TARGET'].mean()
axes[2].bar(range(len(age_default)), age_default.values, color=TEAL, edgecolor='white')
axes[2].axhline(overall_rate, color=GOLD, linestyle='--', linewidth=1.5)
axes[2].set_xticks(range(len(age_default)))
axes[2].set_xticklabels([str(b) for b in age_default.index], rotation=45, ha='right', fontsize=9)
axes[2].set_title('Default Rate by Age Group', color=NAVY, fontweight='bold')
axes[2].set_ylabel('Default Rate')

plt.suptitle('Gender & Age vs Default Risk', fontsize=14, color=NAVY, fontweight='bold')
plt.tight_layout()
plt.show()


<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Age is one of the strongest predictors.</strong> Younger applicants (20–30) have default rates of 10–12%, while older applicants (55+) default at only 4–5%. The relationship is roughly monotonic: older → lower risk. This likely reflects accumulated financial stability, paid-off debts, and more conservative borrowing behaviour. Gender shows a gap too — but be cautious about using it directly in models due to regulatory and ethical considerations.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 3: Numeric Features vs Default</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">How do continuous variables differ between repaid and defaulted loans?</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Use box plots and KDE (density) plots to compare the distribution of numeric features between defaulters and non-defaulters. Identify features where the two groups are clearly separated versus features where they overlap almost completely.</span>
</div>

<div style="background: #E0F2F2; border: 2px solid #008C8C; padding: 16px 22px; border-radius: 8px; margin: 12px 0;">
<strong style="color: #008C8C;">📐 How to Read These Plots</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">
<strong>Box plots</strong> show the median (line), interquartile range (box), and outliers (dots).
If the boxes for Repaid and Default are in very different positions, the feature is discriminative.
If they overlap completely, the feature is weak.<br><br>
<strong>KDE (density) plots</strong> are smoothed histograms. Where the Repaid curve and Default curve
separate, the feature is useful. Where they overlap, it is not.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 4: Box Plot Comparison — Key Numeric Features</h3>
</div>

In [ ]:
# Box Plot Comparison — Repaid vs Default
numeric_features = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DEBT_TO_INCOME',
    'ANNUITY_BURDEN', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AGE_YEARS', 'EMPLOYMENT_YEARS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
]

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
axes = axes.ravel()

group0 = app[app['TARGET'] == 0]
group1 = app[app['TARGET'] == 1]

for i, feat in enumerate(numeric_features):
    q1  = app[feat].quantile(0.01)
    q99 = app[feat].quantile(0.99)
    d0 = group0[feat].dropna().clip(q1, q99)
    d1 = group1[feat].dropna().clip(q1, q99)
    bp = axes[i].boxplot([d0, d1], labels=['Repaid', 'Default'],
                         patch_artist=True, widths=0.5,
                         medianprops=dict(color=GOLD, linewidth=2))
    bp['boxes'][0].set_facecolor(TEAL)
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(CORAL)
    bp['boxes'][1].set_alpha(0.7)
    axes[i].set_title(feat, color=NAVY, fontweight='bold', fontsize=10)

plt.suptitle('Box Plot Comparison: Repaid vs Default', fontsize=14, color=NAVY, fontweight='bold')
plt.tight_layout()
plt.show()


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Use <code>ax.boxplot([group0_data, group1_data], labels=['Repaid','Default'], patch_artist=True)</code>. Clip outliers with <code>data[feat].quantile(0.99)</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Feature discrimination strength:</strong><br>• <strong>Strong separation:</strong> EXT_SOURCE_2, EXT_SOURCE_3, AGE_YEARS — the boxes are clearly shifted between groups.<br>• <strong>Moderate separation:</strong> EMPLOYMENT_YEARS, ANNUITY_BURDEN, DEBT_TO_INCOME — some shift, but lots of overlap.<br>• <strong>Weak separation:</strong> AMT_INCOME_TOTAL, CNT_CHILDREN, CNT_FAM_MEMBERS — boxes almost identical.<br>Features with strong box separation will likely be important in our models. Those with weak separation might still contribute in combination with other features.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 5: KDE Density Overlays — Seeing the Full Shape</h3>
</div>

In [ ]:
# KDE Density Overlays — Top 6 Features
top_features = [
    'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS',
    'EMPLOYMENT_YEARS', 'DEBT_TO_INCOME', 'ANNUITY_BURDEN',
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, feat in enumerate(top_features):
    for target_val, color, label in [(0, TEAL, 'Repaid'), (1, CORAL, 'Default')]:
        subset = app[app['TARGET'] == target_val][feat].dropna()
        q99 = subset.quantile(0.99)
        subset = subset[subset <= q99]
        subset.plot.kde(ax=axes[i], color=color, label=label, linewidth=2.5)
    axes[i].set_title(feat, color=NAVY, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].legend(fontsize=9)
    axes[i].fill_between(axes[i].lines[0].get_xdata(),
                         axes[i].lines[0].get_ydata(), alpha=0.15, color=TEAL)
    axes[i].fill_between(axes[i].lines[1].get_xdata(),
                         axes[i].lines[1].get_ydata(), alpha=0.15, color=CORAL)

plt.suptitle('KDE Density Overlays: Repaid vs Default', fontsize=14, color=NAVY, fontweight='bold')
plt.tight_layout()
plt.show()


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Filter by target: <code>app[app['TARGET']==0][feat].plot.kde(ax=ax, label='Repaid')</code>. The KDE is a smoothed histogram — it shows the probability density.</span>
</div>

<div style="background: #F0E6F6; border-left: 5px solid #6C5B7B; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #6C5B7B;">👔 Manager’s Take</strong><br>
<span style="color: #333;">KDE plots are one of the best tools for communicating feature importance to executives. Where the teal and coral curves pull apart, you can point and say: “This is where the model can distinguish good from bad applicants.” EXT_SOURCE_2 and EXT_SOURCE_3 show the clearest separation — these are the features that should anchor any credit-scoring model.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 4: Outlier Detection & Treatment</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Finding and handling extreme values</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Identify outliers in key numeric features using the IQR method and visual inspection. Understand why outlier treatment is necessary for many models, and when to leave outliers alone.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 6: Outlier Audit</h3>
</div>

In [ ]:
# Outlier Audit using IQR Method
outlier_features = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DEBT_TO_INCOME', 'ANNUITY_BURDEN', 'AGE_YEARS', 'EMPLOYMENT_YEARS', 'CNT_CHILDREN',
]

outlier_report = []
for feat in outlier_features:
    data = app[feat].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((data < lower) | (data > upper)).sum()
    outlier_report.append({
        'Feature': feat,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Lower Fence': round(lower, 2),
        'Upper Fence': round(upper, 2),
        'N Outliers': n_out,
        'Outlier %': round(n_out / len(data) * 100, 2),
    })

outlier_df = pd.DataFrame(outlier_report).sort_values('Outlier %', ascending=False)
print("Outlier Report (IQR Method):")
print(outlier_df.to_string(index=False))

# Winsorise at 99th percentile
for feat in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DEBT_TO_INCOME']:
    cap = app[feat].quantile(0.99)
    app[feat + '_CAPPED'] = app[feat].clip(upper=cap)
    print(f"  Capped {feat} at {cap:,.0f}")
print("\n✅ Winsorisation applied to key financial features.")


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">IQR = Q3 - Q1. Lower fence = Q1 - 1.5*IQR. Upper fence = Q3 + 1.5*IQR. Count outliers: <code>(data < lower).sum() + (data > upper).sum()</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>AMT_INCOME_TOTAL has the most outliers</strong> — some applicants report incomes in the millions. These could be data errors, business clients, or genuinely wealthy individuals. For modelling, we typically <em>cap</em> (winsorise) rather than remove outliers, because extreme income is informative (it signals low default risk). Key rule of thumb: <strong>cap at the 99th percentile</strong> unless you have a domain reason to do otherwise.</span>
</div>

<div style="background: #FDE8E5; border-left: 5px solid #E8634A; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #E8634A;">⚠️ Important</strong><br>
<span style="color: #333;"><strong>When NOT to remove outliers:</strong> In credit risk, extreme values often carry real signal. A person with 10 children or extremely high debt-to-income is genuinely riskier. Removing them removes precisely the cases your model needs to learn from. Prefer <strong>capping (winsorising)</strong> or <strong>log transforms</strong> over deletion.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 5: Feature Transformations</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Making skewed features model-friendly</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Apply log transformations and binning to highly skewed features. Understand <em>why</em> transformations help models learn better, using visual before/after comparisons.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 16px 22px; border-radius: 8px; margin: 12px 0;">
<strong style="color: #B8953E;">📐 Why Transform Features?</strong>
<p style="color: #333; margin-top: 8px; line-height: 1.6;">
Many models (logistic regression, PCA, k-means) assume that features have roughly symmetric distributions.
Highly skewed features (like income, where most people earn modestly but a few earn millions) violate
this assumption and can distort model fitting.<br><br>
<strong>Log transform:</strong> Compresses the right tail. Income of 100K and 1M become 5.0 and 6.0 — much closer.<br>
<strong>Binning:</strong> Converts a continuous feature into ordered categories. Useful when the relationship with
the target is non-linear or step-wise.
</p>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 7: Log Transform — Before and After</h3>
</div>

In [ ]:
# Log Transform — Before and After
skewed_features = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']

fig, axes = plt.subplots(4, 2, figsize=(14, 16))

skew_report = []
for i, feat in enumerate(skewed_features):
    original = app[feat].dropna()
    log_transformed = np.log1p(original)

    # Before
    axes[i][0].hist(original.clip(upper=original.quantile(0.99)),
                    bins=60, color=CORAL, alpha=0.8, edgecolor='white')
    axes[i][0].set_title(f'{feat} — RAW (skew={original.skew():.1f})', color=NAVY, fontweight='bold', fontsize=10)

    # After
    axes[i][1].hist(log_transformed, bins=60, color=TEAL, alpha=0.8, edgecolor='white')
    axes[i][1].set_title(f'log1p({feat}) — skew={log_transformed.skew():.2f}', color=NAVY, fontweight='bold', fontsize=10)

    app[f'LOG_{feat}'] = np.log1p(app[feat])
    skew_report.append({'Feature': feat, 'Skew Before': round(original.skew(), 2),
                        'Skew After': round(log_transformed.skew(), 2)})

plt.suptitle('Log Transform: Before vs After', fontsize=14, color=NAVY, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSkewness Reduction Summary:")
print(pd.DataFrame(skew_report).to_string(index=False))


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Use <code>np.log1p(data)</code> instead of <code>np.log(data)</code> — log1p handles zero values safely. Compute skewness with <code>data.skew()</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;">The log transform dramatically reduces skewness. AMT_INCOME_TOTAL goes from a skew of ~20 to ~1. This means the distribution is now much more symmetric and bell-shaped. For models like logistic regression, this transformation can meaningfully improve performance because the model can now see the full range of income variation, not just “everyone is normal except a few extreme outliers.”</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 8: Intelligent Binning — When Categories Tell a Better Story</h3>
</div>

In [ ]:
# Intelligent Binning — EXT_SOURCE_2 and AMT_INCOME_TOTAL
overall_rate = app['TARGET'].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# EXT_SOURCE_2 — equal-width bins
app['EXT2_BIN'] = pd.cut(app['EXT_SOURCE_2'], bins=10)
ext2_default = app.groupby('EXT2_BIN')['TARGET'].mean().dropna()
axes[0].bar(range(len(ext2_default)), ext2_default.values,
            color=[CORAL if r > overall_rate else TEAL for r in ext2_default.values],
            edgecolor='white')
axes[0].axhline(overall_rate, color=GOLD, linestyle='--', linewidth=1.8, label=f'Overall: {overall_rate:.2%}')
axes[0].set_xticks(range(len(ext2_default)))
axes[0].set_xticklabels([str(b) for b in ext2_default.index], rotation=45, ha='right', fontsize=8)
axes[0].set_title('Default Rate by EXT_SOURCE_2 Decile', color=NAVY, fontweight='bold')
axes[0].set_ylabel('Default Rate')
axes[0].legend()

# AMT_INCOME_TOTAL — quantile-based bins
app['INCOME_QBIN'] = pd.qcut(app['AMT_INCOME_TOTAL'], q=5,
                              labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (Highest)'])
income_default = app.groupby('INCOME_QBIN', observed=True)['TARGET'].mean()
axes[1].bar(range(len(income_default)), income_default.values,
            color=[CORAL if r > overall_rate else TEAL for r in income_default.values],
            edgecolor='white')
axes[1].axhline(overall_rate, color=GOLD, linestyle='--', linewidth=1.8)
axes[1].set_xticks(range(len(income_default)))
axes[1].set_xticklabels(income_default.index, rotation=20, ha='right', fontsize=9)
axes[1].set_title('Default Rate by Income Quintile', color=NAVY, fontweight='bold')
axes[1].set_ylabel('Default Rate')

plt.suptitle('Binning Analysis: EXT_SOURCE_2 vs Income', fontsize=14, color=NAVY, fontweight='bold')
plt.tight_layout()
plt.show()


<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;"><strong>EXT_SOURCE_2 shows a near-perfect monotonic relationship:</strong> as the score increases, default rate drops from ~18% (lowest decile) to ~3% (highest decile). This is exactly the kind of feature that makes a model’s job easy. Income, by contrast, shows a much flatter pattern — higher income helps, but the effect is weaker and less consistent.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 6: Cross-Feature Interactions</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">When two features together tell a richer story</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Explore whether combining two features reveals patterns invisible in either feature alone. This is the first step toward feature engineering — the art of creating new variables.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 9: Age × Income Interaction</h3>
</div>

In [ ]:
# Age x Income Interaction Heatmap
app['AGE_GROUP'] = pd.cut(app['AGE_YEARS'],
                           bins=[20, 30, 40, 50, 60, 70],
                           labels=['20-30', '30-40', '40-50', '50-60', '60-70'])
app['INCOME_GROUP'] = pd.qcut(app['AMT_INCOME_TOTAL'], q=5,
                               labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

pivot = app.pivot_table(values='TARGET', index='AGE_GROUP',
                        columns='INCOME_GROUP', aggfunc='mean', observed=True)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, vmin=0.04, vmax=0.15,
            cbar_kws={'label': 'Default Rate'})
ax.set_title('Default Rate by Age Group x Income Quintile\n(Young + Low Income = Highest Risk)',
             color=NAVY, fontweight='bold', fontsize=13)
ax.set_xlabel('Income Group')
ax.set_ylabel('Age Group')
plt.tight_layout()
plt.show()


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Use <code>pd.cut()</code> for age bins, <code>pd.qcut()</code> for income quintiles, and <code>app.pivot_table(values='TARGET', index='AGE_GROUP', columns='INCOME_GROUP', aggfunc='mean')</code>.</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Young + Low Income = Highest Risk.</strong> The top-left corner of the heatmap (young, low income) shows default rates of 12–14%, while the bottom-right (older, high income) shows 3–5%. This interaction effect is <em>stronger</em> than either feature alone. In Session 14, we’ll formalise this kind of analysis with correlation screening and dimensionality reduction.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 10: External Score × Debt-to-Income</h3>
</div>

In [ ]:
# EXT_SOURCE_2 x Debt-to-Income Scatter — The Risk Landscape
sample = app[['EXT_SOURCE_2', 'DEBT_TO_INCOME', 'TARGET']].dropna().sample(10000, random_state=42)
sample = sample[sample['DEBT_TO_INCOME'] <= 15]

fig, ax = plt.subplots(figsize=(11, 7))
for target_val, color, label, alpha in [(0, TEAL, 'Repaid', 0.3), (1, CORAL, 'Default', 0.6)]:
    subset = sample[sample['TARGET'] == target_val]
    ax.scatter(subset['EXT_SOURCE_2'], subset['DEBT_TO_INCOME'],
               c=color, label=label, alpha=alpha, s=12, edgecolors='none')

ax.axvline(0.3, color=GOLD, linestyle='--', linewidth=1.2, alpha=0.7, label='EXT_SOURCE_2 = 0.3')
ax.axhline(5, color=NAVY, linestyle='--', linewidth=1.2, alpha=0.7, label='DTI = 5')
ax.set_xlabel('EXT_SOURCE_2 (Higher = More Creditworthy)')
ax.set_ylabel('Debt-to-Income Ratio')
ax.set_title('Risk Landscape: EXT_SOURCE_2 vs Debt-to-Income\nDanger Zone = Bottom-Left (Low Score + High Leverage)',
             color=NAVY, fontweight='bold')
ax.legend(fontsize=10)
ax.annotate('DANGER ZONE', xy=(0.1, 12), fontsize=11, color=CORAL,
            fontweight='bold', alpha=0.8)
ax.annotate('SAFE ZONE', xy=(0.65, 1), fontsize=11, color=TEAL,
            fontweight='bold', alpha=0.8)
plt.tight_layout()
plt.show()


<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;">The scatter plot reveals a clear <strong>risk landscape</strong>: low external score + high leverage = concentrated defaults (bottom-left cluster of red dots). High external score + moderate leverage = almost no defaults. This 2D view is more powerful than either feature alone and previews the kind of multi-feature thinking that underpins all credit-scoring models.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 7: Correlation Landscape</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">A visual, intuitive introduction to feature relationships</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Build a correlation matrix for key numeric features and learn to read it. Identify which features are correlated with TARGET (useful for prediction) and which features are correlated with <em>each other</em> (potential redundancy).</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 11: Correlation with TARGET</h3>
</div>

In [ ]:
# Correlation with TARGET — Top 15 & Bottom 15
target_corr = (app.select_dtypes(include=[np.number])
               .corr()['TARGET']
               .drop('TARGET')
               .sort_values())

top15    = target_corr.tail(15)
bottom15 = target_corr.head(15)
combined = pd.concat([bottom15, top15])

fig, ax = plt.subplots(figsize=(12, 10))
colors = [TEAL if c < 0 else CORAL for c in combined.values]
bars = ax.barh(combined.index, combined.values, color=colors, edgecolor='white', height=0.7)
ax.axvline(0, color=NAVY, linewidth=1)
ax.set_title('Top 15 Positive & Negative Correlations with TARGET',
             color=NAVY, fontweight='bold', fontsize=13)
ax.set_xlabel('Pearson Correlation with TARGET')
for bar, val in zip(bars, combined.values):
    ax.text(val + (0.001 if val >= 0 else -0.001),
            bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.show()

print("\nTop 5 PROTECTIVE features (negative correlation):")
print(target_corr.head(5).to_string())
print("\nTop 5 RISKY features (positive correlation):")
print(target_corr.tail(5).to_string())


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Get all correlations: <code>app.select_dtypes(include=[np.number]).corr()['TARGET']</code>. Sort: <code>.sort_values()</code>. Bottom 15 = head(15), top 15 = tail(15).</span>
</div>

<div style="background: linear-gradient(90deg, #E0F2F2, #FDF6E8); border-left: 5px solid #008C8C; border-right: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🧠 Interpretation</strong><br>
<span style="color: #333;"><strong>Negative correlation = protective (lower default).</strong> EXT_SOURCE_3 (≈0.18), EXT_SOURCE_2 (≈0.16), and AGE_YEARS (≈0.08) are the strongest protective factors. <strong>Positive correlation = risky.</strong> DAYS_BIRTH (which is negative, so higher values = younger = riskier) and some FLAG_DOCUMENT columns show positive correlation. Most features have very weak correlation (<0.05 in absolute terms). This is normal — credit default is a complex event that no single feature predicts well.</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 12: Feature-to-Feature Correlation Heatmap</h3>
</div>

In [ ]:
# Feature-to-Feature Correlation Heatmap
heatmap_features = [
    'TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AGE_YEARS', 'EMPLOYMENT_YEARS', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
    'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DEBT_TO_INCOME', 'ANNUITY_BURDEN',
    'CREDIT_GOODS_RATIO', 'EXT_SCORE_BLEND', 'DAYS_EMPLOYED_ANOMALY', 'CNT_CHILDREN',
]

corr_matrix = app[heatmap_features].corr()

# Upper triangle mask
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn_r', center=0, linewidths=0.5,
            ax=ax, square=True, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap\n(First row = correlation with TARGET)',
             color=NAVY, fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Flag highly correlated pairs
print("\nHighly correlated feature pairs (|r| > 0.7, excluding TARGET):")
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        f1, f2 = corr_matrix.columns[i], corr_matrix.columns[j]
        if f1 == 'TARGET' or f2 == 'TARGET':
            continue
        r = corr_matrix.loc[f1, f2]
        if abs(r) > 0.7:
            high_corr.append({'Feature 1': f1, 'Feature 2': f2, 'Correlation': round(r, 3)})
print(pd.DataFrame(high_corr).to_string(index=False))


<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;"><strong>Two things to notice:</strong><br>• <strong>Feature-target correlations</strong> (first column): EXT_SOURCE variables dominate. Everything else is weaker.<br>• <strong>Feature-feature correlations</strong>: AMT_CREDIT and AMT_GOODS_PRICE are highly correlated (r≈0.97). DEBT_TO_INCOME and CREDIT_GOODS_RATIO are also correlated. These <em>redundant</em> features will be handled in Session 14 using variance filters and PCA.</span>
</div>

<div style="background: linear-gradient(135deg, #E8EDF4 0%, #E0F2F2 100%); border: 2px solid #008C8C; padding: 18px 22px; border-radius: 10px; margin: 20px 0;">
<strong style="color: #0B1F3F;">🔀 Why This Matters Next</strong><br>
<span style="color: #333; line-height: 1.6;">In Session 14, we’ll formalise what we’ve seen here. The correlation heatmap will become a tool for <strong>multicollinearity detection</strong>. The feature-target correlations will feed into <strong>variance filters</strong>. And the patterns we noticed (groups of related features) will motivate <strong>PCA and Factor Analysis</strong> — mathematical techniques for discovering the underlying “themes” hidden in 120+ features.</span>
</div>

<div style="background: linear-gradient(90deg, #0B1F3F, #008C8C); padding: 14px 24px; border-radius: 8px; margin: 30px 0 12px 0;">
<h2 style="color: white; margin: 0; font-family: Georgia, serif;">Part 8: Feature Scorecard</h2>
<p style="color: #008C8C; margin: 4px 0 0 0; font-size: 1.05em;">Ranking features by their apparent predictive power</p>
</div>

<div style="background: #E8EDF4; border-left: 5px solid #0B1F3F; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #0B1F3F;">🎯 Learning Objective</strong><br>
<span style="color: #333;">Synthesise everything we have learned into a single scorecard that ranks features by their usefulness for predicting default. This is the deliverable that bridges data exploration (Sessions 13 + 13B) to formal modelling (Sessions 14+).</span>
</div>

<div style="background: #FDF6E8; border: 2px solid #B8953E; padding: 10px 18px; border-radius: 8px; margin: 20px 0 8px 0;">
<h3 style="color: #0B1F3F; margin: 0;">✏️ Exercise 13: Build the Feature Scorecard</h3>
</div>

In [ ]:
# Feature Scorecard
numeric_cols_clean = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AGE_YEARS', 'EMPLOYMENT_YEARS',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DEBT_TO_INCOME', 'ANNUITY_BURDEN', 'CREDIT_GOODS_RATIO',
    'EXT_SCORE_BLEND', 'DAYS_EMPLOYED_ANOMALY',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
    'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY',
]

scorecard = []
g0 = app[app['TARGET'] == 0]
g1 = app[app['TARGET'] == 1]

for col in numeric_cols_clean:
    if col not in app.columns:
        continue
    # 1. Absolute correlation with TARGET
    corr = abs(app[col].corr(app['TARGET']))

    # 2. Cohen's d
    m0, m1 = g0[col].mean(), g1[col].mean()
    s0, s1 = g0[col].std(),  g1[col].std()
    pooled_std = np.sqrt((s0**2 + s1**2) / 2)
    cohens_d = abs(m0 - m1) / pooled_std if pooled_std > 0 else 0

    # 3. Missing %
    missing_pct = app[col].isnull().mean() * 100

    # 4. Signal Strength = corr * 10 + cohens_d * 5  (weighted composite)
    signal = round(corr * 10 + cohens_d * 5, 3)

    scorecard.append({
        'Feature': col,
        'Abs Corr': round(corr, 4),
        "Cohen's d": round(cohens_d, 4),
        'Missing %': round(missing_pct, 1),
        'Signal Strength': signal,
    })

scorecard_df = (pd.DataFrame(scorecard)
                .sort_values('Signal Strength', ascending=False)
                .reset_index(drop=True))
scorecard_df.index += 1
print("Feature Scorecard — Ranked by Signal Strength:")
print(scorecard_df.to_string())


<div style="background: #FDF6E8; border-left: 5px solid #B8953E; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #B8953E;">💡 Hint</strong><br>
<span style="color: #555;">Cohen’s d measures how far apart two group means are, relative to their variability. <code>d = abs(mean0 - mean1) / sqrt((std0**2 + std1**2) / 2)</code>. A d > 0.2 is “small”, > 0.5 is “medium”, > 0.8 is “large”.</span>
</div>

In [ ]:
# Scorecard Visualisation
def get_color(score):
    if score > 5:  return TEAL
    elif score > 2: return GOLD
    else:           return CORAL

top20 = scorecard_df.head(20).sort_values('Signal Strength', ascending=True)
colors = [get_color(s) for s in top20['Signal Strength']]

fig, ax = plt.subplots(figsize=(12, 9))
bars = ax.barh(top20['Feature'], top20['Signal Strength'],
               color=colors, edgecolor='white', height=0.7)
ax.set_title('Feature Scorecard — Top 20 by Signal Strength',
             color=NAVY, fontweight='bold', fontsize=13)
ax.set_xlabel('Signal Strength Score (Corr x 10 + Cohen\'s d x 5)')
for bar, score in zip(bars, top20['Signal Strength']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{score:.2f}', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=TEAL,  label='Strong  (> 5)'),
    Patch(facecolor=GOLD,  label='Moderate (2–5)'),
    Patch(facecolor=CORAL, label='Weak     (< 2)'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()


<div style="background: #E0F2F2; border-left: 5px solid #008C8C; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">
<strong style="color: #008C8C;">📊 Business Insight</strong><br>
<span style="color: #333;"><strong>The scorecard confirms what we’ve seen throughout this session:</strong><br>• <strong>Tier 1 (Strong):</strong> EXT_SOURCE_2, EXT_SOURCE_3, EXT_SCORE_BLEND, AGE_YEARS — these are the pillars of any model.<br>• <strong>Tier 2 (Moderate):</strong> EMPLOYMENT_YEARS, REGION_RATING, DAYS_ID_PUBLISH, ANNUITY_BURDEN — valuable supporting features.<br>• <strong>Tier 3 (Weak):</strong> AMT_INCOME_TOTAL, CNT_CHILDREN, CNT_FAM_MEMBERS — individually weak, but may contribute in ensembles.<br>This ranking will guide feature selection in Session 14 and model building in Sessions 15+.</span>
</div>

In [ ]:
SEP68 = "─" * 68
print(f"\n┌{SEP68}┐")
for label, value in findings:
    print(f"│  {label:<25s} {value:>40s} │")
print(f"└{SEP68}┘")

print("\n✅ Session 13B Data Deep Dive Complete!")
print("➡️  Next: Session 14 — Signal Extraction (PCA, Factor Analysis, Variance Filters)")


<div style="background: linear-gradient(135deg, #0B1F3F 0%, #008C8C 100%); padding: 30px; border-radius: 12px; margin-top: 30px;">
<h2 style="color: white; font-family: Georgia, serif; margin: 0;">
✅ Session 13B Complete
</h2>
<p style="color: #B8953E; font-size: 1.15em; margin-top: 10px;">
You now have deep intuition about which features predict default and why. You can articulate feature
importance using default rate charts, KDE plots, and correlation analysis. You have a ranked feature
scorecard that will guide all subsequent modelling work.
</p>
<p style="color: #B0D0D0; font-size: 1em; margin-top: 8px;">
<strong>Next Session Preview:</strong> In Session 14, we’ll formalise this intuition with
<strong>correlation screening</strong> (removing redundant features), <strong>variance filters</strong>
(removing uninformative features), and <strong>PCA vs Factor Analysis</strong> (discovering the hidden
structure in 120+ features). The scorecard you built today is your baseline — Session 14 will
show you whether the math agrees with your gut.
</p>
</div>